# Production-Ready Reinforcement Learning

Let's time to bridge the gap between theoretical knowledge and real-world application. This notebook
introduces the essential techniques and tools for making RL agents production-ready.

### What We'll Cover:

1.  **Code Structure & Refactoring**: How to structure code to be modular and reusable.
2.  **Metrics & Monitoring**: Using Tensorboard to track losses, rewards, and gradients.
3.  **Checkpointing**: Saving and loading models to resume training or for inference.
4.  **Debugging & Troubleshooting**: Common techniques for when your RL agent isn't learning.
5.  **The Importance of Multiple Seeds**: Ensuring your results are robust and reproducible.
6.  **Scaling with Parallelization (Ray)**: Speeding up training with parallel environments.
7.  **Automated Hyperparameter Tuning (Optuna)**: Finding the best hyperparameters automatically.
8.  **What Else?**: A look at MLOps, CI/CD, Safe RL, and other frontiers.

Let's dive in!


### Imports and Setup

First, let's import the necessary libraries. We'll be using PyTorch, Gymnasium, and several new
libraries for our production techniques.


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from torch.utils.tensorboard import SummaryWriter

import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random
import os
import ray
import optuna
import copy

from util.gymnastics import gym_simulation, init_random

In [ ]:
RUNS_DIR = "runs"
CHECKPOINTS_DIR = "checkpoints"

os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

### A Note on Environment Choice: CartPole

`CartPole` provides a **dense reward** (+1 for every timestep the pole is balanced), which allows
our agent to learn quickly. This rapid feedback loop makes it much easier to see the effects of our
tooling and techniques without waiting a long time for the agent to solve a complex exploration
problem.


In [ ]:
ENV_NAME = "CartPole-v1"
TEMP_ENV = gym.make(ENV_NAME)
STATE_DIM = TEMP_ENV.observation_space.shape[0]
ACTION_DIM = TEMP_ENV.action_space.n
TEMP_ENV.close()

In [ ]:
gym_simulation(ENV_NAME)

---


### Part 1: Modular Code Structure - An A2C Trainer

A key principle of production-ready code is **modularity**. Instead of rewriting the training loop
in every section, we will encapsulate the logic into a reusable `A2C_Trainer` class. This makes the
code cleaner, easier to maintain, and less prone to errors.

Our trainer will handle:

- The Actor-Critic model and optimizer.
- A single step of the A2C algorithm.
- Logic for running a full episode.


In [ ]:
class ActorCritic(nn.Module):
    """The Actor-Critic neural network model."""

    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(ActorCritic, self).__init__()
        # TODO: Create a Sequential module for the actor: linear, relu, linear, softmax
        self.actor = None
        # TODO: Create a Sequential module for the critic: linear, relu, linear
        self.critic = None

    def forward(self, state):
        # TODO: Pass the state through the actor to get the policy
        policy = None
        # TODO: Pass the state through the critic to get the value
        value = None
        return policy, value

    @torch.no_grad
    def act(self, state):
        # TODO: Implement the act method for inference
        # 1. Convert state to a tensor and add a batch dimension
        # 2. Get the policy from the model
        # 3. Return the action with the highest probability (use torch.argmax)
        return 0  # Placeholder


class A2C_Trainer:
    """A trainer class to encapsulate the A2C training logic."""

    def __init__(self, state_dim, action_dim, lr=0.001, gamma=0.99, hidden_dim=128):
        self.model = ActorCritic(state_dim, action_dim, hidden_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.gamma = gamma

    def train_step(self, log_probs, values, rewards, entropies):
        """Performs a single training update and returns diagnostics."""
        # TODO: Calculate discounted returns (R). Hint: Iterate backwards through rewards.
        returns = []

        # TODO: Convert returns and values to tensors. Squeeze values.
        returns = None
        values = None

        # TODO: Calculate advantages. Detach values to stop gradients.
        advantages = None

        # TODO: Calculate actor loss.
        actor_loss = None

        # TODO: Calculate critic loss using Mean Squared Error.
        critic_loss = None

        # TODO: Calculate the total loss.
        loss = None

        # TODO: Perform backpropagation.
        # 1. Zero gradients
        # 2. Backward pass
        # 3. Clip gradients with torch.nn.utils.clip_grad_norm_ (capture the returned total norm)
        # 4. Optimizer step
        grad_norm = None

        # TODO: Return a dictionary of diagnostics: loss, grad_norm, entropy (mean of entropies),
        #       advantage_mean, advantage_std. Use .item() on tensor scalars.
        return {
            "loss": 0.0,
            "grad_norm": 0.0,
            "entropy": 0.0,
            "advantage_mean": 0.0,
            "advantage_std": 0.0,
        }

    def run_episode(self, env):
        """Runs a single episode and collects trajectories.

        The env is seeded once by the caller (via `init_random`); subsequent resets
        here advance the seeded RNG stream, so each episode sees a fresh start.
        """
        state, _ = env.reset()
        done = False
        total_reward = 0
        log_probs, values, rewards, entropies = [], [], [], []

        while not done:
            # TODO: Get policy and value from the model.
            policy, value = None, None

            # TODO: Create a Categorical distribution and sample an action.
            dist = None
            action = None

            # TODO: Step the environment. Gymnasium returns (next_state, reward, terminated,
            #       truncated, info). Production loops must stop on EITHER signal — collapsing
            #       them into `done = terminated or truncated` avoids subtle bugs at time-limit
            #       boundaries.
            next_state, reward, terminated, truncated, _ = env.step(0)  # Placeholder
            done = terminated or truncated

            # TODO: Store the log prob, value, reward, and the distribution's entropy.
            # ...

            state = next_state
            total_reward += reward

        return log_probs, values, rewards, entropies, total_reward

---


### Part 2: Metrics and Monitoring with Tensorboard

**Tensorboard** is a powerful visualization toolkit for understanding, debugging, and optimizing
machine learning experiments. It allows us to log and visualize key metrics in real-time.

Beyond the headline reward curve, a few _diagnostic_ scalars are what actually tell you whether
training is healthy:

- **Reward** — the headline metric; is the agent learning at all?
- **Loss** — policy + critic loss; should trend down (but can be noisy in RL).
- **Policy entropy** — how "undecided" the policy is. Should decay gradually; a crash to zero means
  the policy has collapsed onto one action.
- **Gradient norm** — magnitude of the gradient before clipping. Spikes indicate instability, often
  an LR or reward-scale problem.
- **Advantage mean/std** — the signal driving policy updates. An advantage std that explodes means
  returns and values have drifted apart (critic is lagging or rewards are unscaled).

We log all of these below. We'll lean on them heavily in Part 4 to diagnose a broken run.


In [ ]:
def train_with_monitoring(
    seed, n_episodes=1000, lr=0.001, gamma=0.99, run_name_suffix="", save_best=True
):
    # `init_random(env, seed=...)` seeds every RNG this notebook touches (Python, NumPy,
    # PyTorch CPU+CUDA, the env, the action space) and enables deterministic algorithms.
    env = init_random(gym.make(ENV_NAME), seed=seed)

    # TODO: Create the trainer.
    trainer = None

    run_name = f"a2c_seed_{seed}{run_name_suffix}"
    os.makedirs(os.path.join(CHECKPOINTS_DIR, run_name), exist_ok=True)
    # TODO: Create the SummaryWriter in RUNS_DIR/run_name.
    writer = None

    episode_rewards = []
    best_avg_reward = -np.inf

    for episode in range(n_episodes):
        # TODO: Run an episode with the trainer (returns log_probs, values, rewards,
        #       entropies, total_reward).
        log_probs, values, rewards, entropies, total_reward = None, None, None, None, 0
        # TODO: Perform one training step. It returns a dict of diagnostics.
        stats = None

        episode_rewards.append(total_reward)

        # TODO: Log the core scalars every episode:
        #   "Reward/episode_reward", "Loss/total_loss",
        #   "Diagnostics/entropy", "Diagnostics/grad_norm",
        #   "Diagnostics/advantage_mean", "Diagnostics/advantage_std".
        # Hint: writer.add_scalar(tag, value, episode)

        # Histograms are expensive — log sparsely (e.g., every 50 episodes).
        if episode % 50 == 0:
            for name, param in trainer.model.named_parameters():
                if param.grad is not None:
                    # TODO: Log per-parameter gradient histograms.
                    # Hint: writer.add_histogram(f"Gradients/{name}", param.grad, episode)
                    pass

        if len(episode_rewards) > 100:
            # TODO: Compute the 100-episode moving average and log it as
            #       "Reward/moving_avg_reward".
            avg_reward = None

            if save_best and avg_reward > best_avg_reward:
                best_avg_reward = avg_reward
                # TODO: Save a full checkpoint dict (not just weights) so training
                # can truly resume: keys {'model', 'optimizer', 'episode',
                # 'best_avg_reward'} written to CHECKPOINTS_DIR/run_name/model_best.pth.
                # Hint: torch.save({...}, path)
                pass

    print(f"Finished training for seed {seed}.")
    writer.close()
    env.close()
    return episode_rewards


rewards_seed_1 = train_with_monitoring(seed=42)

# To view the logs, run this command in your terminal:
# tensorboard --logdir=runs

![TensorBoard](assets/14_PROD_tensorboard.png) <br><small>TensorBoard UI. This is what you should
see when multiple seeds are logged.</small>


---


### Part 3: Checkpointing (Saving & Loading Models)

**Checkpointing** is the process of saving the state of your model during training. This is crucial
for several reasons:

- **Resuming Training**: If your training process is interrupted, you can resume from the last saved
  checkpoint instead of starting over. This is why we save the **optimizer state** alongside the
  model — without it, Adam's running averages restart from zero on reload and learning stalls.
- **Inference**: Once you have a trained model, you need to save it to use it later for making
  predictions.
- **Best Model**: You can save the model that achieved the best performance, which might not be the
  one from the very last epoch.

Before trusting a checkpoint, it is worth a 5-line sanity check: save, reload, confirm the loaded
model produces identical outputs. A production team runs this as an automated test in CI.


In [ ]:
def test_checkpoint_roundtrip():
    """Save a model, reload it, check that predictions match exactly."""
    model = ActorCritic(STATE_DIM, ACTION_DIM)
    sample = torch.randn(1, STATE_DIM)
    with torch.no_grad():
        policy_before, value_before = model(sample)

    path = os.path.join(CHECKPOINTS_DIR, "_roundtrip_test.pth")
    # TODO: Save a checkpoint dict containing the model's state_dict under the key 'model'.

    reloaded = ActorCritic(STATE_DIM, ACTION_DIM)
    # TODO: Load the checkpoint and restore `reloaded`'s weights from the 'model' key.

    with torch.no_grad():
        policy_after, value_after = reloaded(sample)

    # TODO: Assert the outputs match (use torch.allclose).

    os.remove(path)
    print("✅ Checkpoint round-trip test passed!")


test_checkpoint_roundtrip()

In [ ]:
best_model_path = f"{CHECKPOINTS_DIR}/a2c_seed_42/model_best.pth"
agent_model = ActorCritic(STATE_DIM, ACTION_DIM)
# TODO: torch.load the checkpoint, then load the weights from its 'model' key into
# agent_model. Print the saved episode number and best_avg_reward for context.

# TODO: Set the model to evaluation mode. CRITICAL for inference.
# Hint: agent_model.eval()

gym_simulation(ENV_NAME, agent_model)

---


### Part 4: Debugging and Troubleshooting in RL

Debugging RL is notoriously hard: a buggy agent rarely crashes — it just _doesn't learn_. The
diagnostics we logged in Part 2 are exactly how you tell the difference. Let's induce a classic
failure — **too-high learning rate** — and watch what each metric says.


In [ ]:
# Classic RL failure: LR too high. Policy updates overshoot, the critic can't keep up,
# and learning never takes off. Short run (150 episodes) — just enough for the TensorBoard
# signatures to be visible. Launch `tensorboard --logdir=runs` and compare
# `a2c_seed_42` (healthy) against `a2c_seed_42_broken_lr` (broken) side by side.
_ = train_with_monitoring(
    seed=42, n_episodes=150, lr=0.1, run_name_suffix="_broken_lr", save_best=False
)

The broken run shows a few telltale signs that generalize to other bugs. Keep this table handy:

| Symptom in TensorBoard                     | Likely cause                                                        |
| ------------------------------------------ | ------------------------------------------------------------------- |
| Flat reward, entropy ≈ `log(n_actions)`    | No learning signal — check reward scale, advantage sign             |
| Entropy collapses to ~0 early              | Policy collapsed onto one action — add entropy bonus, lower LR      |
| Grad-norm constantly at the clip threshold | LR too high, or rewards unscaled — gradients are exploding          |
| Advantage std growing unboundedly          | Critic is lagging — lower critic LR or normalize returns            |
| Critic loss flat or rising                 | Value head mis-scaled or LR wrong — check critic in isolation       |
| Reward climbs, then crashes                | Overshoot — lower LR, shorten rollouts, or add trust region (→ PPO) |

#### General debugging tips

- **Start simple.** A new algorithm goes on CartPole first — if it doesn't solve CartPole in a few
  minutes, it has a bug.
- **Sanity-check model outputs.** Before training, feed a dummy state through the model and assert
  that `policy.sum() == 1` and `policy.shape == (1, ACTION_DIM)`.
- **Read the paper carefully.** Small implementation details (normalization, advantage computation,
  gradient accumulation) quietly make or break RL.


---


### Part 5: The Importance of Multiple Seeds

A single-seed result says almost nothing about an algorithm. Weight initialization, action sampling,
and environment randomness all interact — a lucky seed can solve CartPole in 200 episodes while an
unlucky one plateaus at 100 reward. The only honest way to report RL results is to run **multiple
seeds** and show mean ± std.


In [ ]:
def plot_rewards(rewards_list):
    """Plots the mean and standard deviation of rewards from multiple seeds."""
    plt.figure(figsize=(12, 6))

    rewards_array = np.array(rewards_list)
    mean_rewards = np.mean(rewards_array, axis=0)
    std_rewards = np.std(rewards_array, axis=0)

    plt.plot(mean_rewards, label="Mean Reward", color="blue")
    plt.fill_between(
        range(len(mean_rewards)),
        mean_rewards - std_rewards,
        mean_rewards + std_rewards,
        color="blue",
        alpha=0.2,
        label="Standard Deviation",
    )

    plt.title("A2C Performance on CartPole-v1 (3 Seeds)")
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.legend()
    plt.grid(True)
    plt.show()


seeds = [42, 123, 789]
all_rewards = []
for seed in seeds:
    rewards = train_with_monitoring(seed=seed, n_episodes=500)
    all_rewards.append(rewards)

plot_rewards(all_rewards)

Notice that the shaded region is wide early on: individual seeds diverge dramatically before
converging. A paper that reports a single lucky seed makes this variance invisible.

**Reproducibility footnote.** "Use multiple seeds" is necessary but not sufficient. A reproducible
RL experiment seeds _every_ RNG source:

- Python's `random`,
- NumPy (`np.random`),
- PyTorch on CPU (`torch.manual_seed`) **and** CUDA (`torch.cuda.manual_seed_all`),
- the environment (`env.reset(seed=...)` on first reset),
- the environment's action space (`env.action_space.seed(...)`),
- any dataloader or sampler.

In this project, `util.gymnastics.init_random(env, seed=...)` is the one-stop helper — it covers all
of the above and also enables `torch.use_deterministic_algorithms(True)` + `cudnn.deterministic`.
Finally: **always evaluate on seeds the agent did not train on** — otherwise you are measuring
memorization, not policy quality.


---


### Part 6: Scaling with Parallelization (Ray)

#### How does Ray work?

**Ray** is a framework for distributed computing that makes it simple to scale Python applications
across multiple cores or machines. Its core philosophy is to turn regular Python functions and
classes into distributable, asynchronous tasks.

We will use two fundamental primitives from Ray Core:

1.  `@ray.remote`: A decorator that turns a Python class or function into a remote object or task
    that can be executed on a separate worker process.
2.  `ray.put()` and `ray.get()`: These functions are used to efficiently transfer objects (like our
    model's weights) to Ray's distributed object store and retrieve results from our remote workers.

Our strategy will be to create several remote **`RolloutWorker`** actors. The main training loop
will send the latest model weights to these workers, who will then independently collect experience
(run episodes) in parallel. The main loop then gathers this experience to perform a single, larger
update to the model.

Two design notes. **Sync vs. async:** we gather all workers before each update (synchronous), which
keeps A2C on-policy at the cost of idle workers waiting on the slowest trajectory. Async
alternatives ([IMPALA](https://arxiv.org/abs/1802.01561), [Ape-X](https://arxiv.org/abs/1803.00933))
feed experience into a replay buffer and train continuously, but require off-policy corrections like
V-trace. **Why `ray.put` the weights:** shipping the full model object to every worker each batch
would be wasteful; the object store lets workers deserialize once and reference shared bytes.
Workers stay stateless w.r.t. the model — only the environment lives on them.


In [ ]:
@ray.remote
class RolloutWorker:
    """A remote actor for collecting experience in parallel."""

    def __init__(self, env_name, seed):
        self.env = gym.make(env_name)
        # Seed once at worker start so subsequent resets advance the RNG stream —
        # otherwise every episode replays the identical starting state.
        self.env.reset(seed=seed)

    def run_episode(self, model_weights):
        state_dim = self.env.observation_space.shape[0]
        action_dim = self.env.action_space.n
        model = ActorCritic(state_dim, action_dim)
        model.load_state_dict(model_weights)

        states, actions, rewards = [], [], []
        # No seed here — we continue the RNG stream set in __init__.
        state, _ = self.env.reset()
        done = False

        while not done:
            # TODO: Sample an action from the model (no gradients needed for workers).
            # Hint: wrap in torch.no_grad(), build a Categorical, sample.
            action = None

            # TODO: Step the env. Use Gymnasium's 5-tuple (terminated, truncated) and stop on
            #       EITHER signal — `done = terminated or truncated`. Running past truncation
            #       inflates batch reward statistics.
            next_state, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated

            # TODO: Store the state, action, and reward.

            state = next_state

        return states, actions, rewards

    def close(self):
        """Closes the worker's environment."""
        self.env.close()


def train_a2c_with_ray(n_batches=250, n_workers=4, lr=0.001, gamma=0.99):
    trainer = A2C_Trainer(STATE_DIM, ACTION_DIM, lr=lr, gamma=gamma)
    writer = SummaryWriter(f"{RUNS_DIR}/a2c_ray")

    # TODO: Create `n_workers` remote workers using the RolloutWorker class.
    # Hint: [RolloutWorker.remote(ENV_NAME, seed=i) for i in range(n_workers)]
    workers = []

    for batch_idx in range(n_batches):
        # TODO: Put the model's state_dict into Ray's object store (ray.put).
        model_weights_id = None

        # TODO: Call run_episode remotely on all workers, then ray.get the results.
        futures = []
        results = []

        batch_reward = 0
        trainer.optimizer.zero_grad()

        for states, actions, rewards in results:
            batch_reward += sum(rewards)

            # TODO: Accumulate gradients from this trajectory into trainer.model.
            # Steps:
            #  1. Convert states/actions to tensors.
            #  2. Re-evaluate policy, values on trainer.model (needed for grad_fn).
            #  3. Compute log_probs via Categorical(policies).log_prob(actions).
            #  4. Compute discounted returns and advantages (detach values).
            #  5. actor_loss + 0.5 * critic_loss, scaled by 1/n_workers.
            #  6. loss.backward()  # accumulates into trainer.model.grad
            pass

        # TODO: One optimizer step after gradients from all workers are accumulated.

        avg_reward = batch_reward / n_workers
        writer.add_scalar("Reward/avg_episode_reward", avg_reward, batch_idx * n_workers)
        if (batch_idx + 1) % 25 == 0:
            print(f"Batch {batch_idx+1}/{n_batches}, Avg Reward: {avg_reward:.2f}")

    writer.close()
    print("Closing remote worker environments...")
    ray.get([w.close.remote() for w in workers])
    print("Ray training finished.")


if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)

train_a2c_with_ray()

ray.shutdown()

---


### Part 7: Automated Hyperparameter Tuning (Optuna)

#### How does Optuna work?

**Optuna** is a hyperparameter optimization framework that automates the search for the best model
settings. It uses a "define-by-run" API that makes it highly flexible and Pythonic.

The core concepts we will use are:

1.  **Study**: A `study` object manages an entire optimization task. We define the goal (e.g.,
    `direction='maximize'`).
2.  **Trial**: A `trial` represents a single execution of our training process with a specific set
    of hyperparameters. Inside our objective function, we ask the `trial` object to `suggest` values
    for each hyperparameter (e.g., `trial.suggest_float('lr', ...)`).
3.  **Objective Function**: This is a function that Optuna will call repeatedly. It takes a `trial`
    object as input, runs our training, and returns a performance score (e.g., the average reward),
    which Optuna then tries to maximize or minimize.

Optuna uses intelligent sampling algorithms (like TPE) to choose which hyperparameter combinations
to try next, making it much more efficient than a simple grid search.


In [ ]:
def objective(trial):
    """The objective function for Optuna to optimize."""
    # TODO: Suggest hyperparameters using the trial object.
    #   lr: float between 1e-4 and 1e-2 (log scale)        -> trial.suggest_float(..., log=True)
    #   gamma: float between 0.9 and 0.999 (log scale)     -> trial.suggest_float(..., log=True)
    #   hidden_dim: categorical from [64, 128, 256]        -> trial.suggest_categorical(...)
    lr = 0.001
    gamma = 0.99
    hidden_dim = 128

    # Multi-seed objective: a single-seed score rewards lucky seeds, not good
    # hyperparameters. Averaging across a handful of seeds filters out most of the
    # noise at the cost of linearly more compute per trial. For a quick demo we use 2.
    seeds = [0, 1]
    seed_scores = []
    for seed_idx, seed in enumerate(seeds):
        env = init_random(gym.make(ENV_NAME), seed=seed)
        trainer = A2C_Trainer(STATE_DIM, ACTION_DIM, lr, gamma, hidden_dim)

        episode_rewards = []
        for episode in range(150):
            # TODO: Run an episode with the trainer.
            log_probs, values, rewards, entropies, total_reward = None, None, None, None, 0
            # TODO: trainer.train_step(...) with all four sequences.
            episode_rewards.append(total_reward)

            # Prune during the first seed only: keeps `step` semantics consistent
            # across trials. Bad HPs get killed fast; survivors pay the full 2-seed cost.
            if seed_idx == 0:
                # TODO: Report the recent mean reward to the trial and check for pruning.
                # Hint: trial.report(np.mean(episode_rewards[-50:]), step=episode)
                # Hint: if trial.should_prune(): raise optuna.exceptions.TrialPruned()
                pass

        seed_scores.append(np.mean(episode_rewards[-50:]))
        env.close()

    # TODO: Return the mean of seed_scores as a float.
    return 0.0


# TODO: Create an Optuna study with direction='maximize' and a MedianPruner.
study = None

# TODO: Run study.optimize with n_trials=10 and timeout=180.

print(f"Best trial value: {study.best_trial.value}")
print(f"Best params: {study.best_params}")

if optuna.visualization.is_available():
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()
    fig2 = optuna.visualization.plot_param_importances(study)
    fig2.show()

---


### What Else?

We've covered the core hands-on tools: modular code, monitoring, checkpointing, debugging,
multi-seed validation, parallelization, and hyperparameter tuning. A real production RL system
reaches further — here are directions worth exploring next.

- **Experiment tracking alternatives.** [Weights & Biases](https://wandb.ai/) and
  [MLflow](https://mlflow.org/) offer richer tracking than TensorBoard, with remote storage,
  artifact versioning, and (for MLflow) a model registry + deployment story.
- **Configuration management.** Hard-coded hyperparameters don't scale past a handful of
  experiments. [Hydra](https://hydra.cc/) and [OmegaConf](https://omegaconf.readthedocs.io/) let you
  compose YAML configs and sweep cleanly.
- **Beyond Optuna.** [Population-Based Training (PBT)](https://arxiv.org/abs/1711.09846) mutates
  hyperparameters _during_ training — a natural fit for RL where optimal HPs drift as the policy
  improves.
- **Scaling beyond Ray.** [Isaac Lab](https://isaac-sim.github.io/IsaacLab/) and
  [EnvPool](https://github.com/sail-sg/envpool) provide GPU-accelerated or C++-vectorized
  environments that dwarf Ray's speedups for compute-bound simulators.
- **MLOps end-to-end.** CI/CD pipelines automate test → train → evaluate → promote.
  [Kubeflow](https://www.kubeflow.org/) orchestrates this on Kubernetes; specialized platforms like
  [Determined AI](https://www.determined.ai/) target ML workloads directly.
- **Safe RL.** In robotics or autonomous systems, a wrong exploratory action can be dangerous.
  Constrained MDPs, safety layers that veto unsafe actions, and recovery policies keep exploration
  within acceptable bounds.

These are all rabbit holes worth falling into once the core loop clicks. Happy training! 🚀
